In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

os.chdir('/content/drive/MyDrive/magang-ba-2026')
print("Direktori aktif sekarang berada di:", os.getcwd())

Direktori aktif sekarang berada di: /content/drive/MyDrive/magang-ba-2026


**Menjalankan src/evaluate_test.py dan Menyimpan Hasil Test Final**

In [3]:
import json
import os
import time
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from transformers import pipeline

# 1. Muat model terpilih dari folder selected (T3)
print("Memuat model terkunci dari models/selected/model_card.json...")
with open("models/selected/model_card.json", "r", encoding="utf-8") as f:
    selected_card = json.load(f)

Memuat model terkunci dari models/selected/model_card.json...


In [4]:
# 2. Muat dataset & split konsisten untuk mengambil test_df
df_comments = pd.read_csv("data/processed/comments_v1.csv")
text_col = 'text_original'
label_col = 'sentiment_label'

df_clean = df_comments.dropna(subset=[text_col, label_col]).copy()
df_clean['reference_label'] = df_clean[label_col]

train_val, test_df = train_test_split(
    df_clean, test_size=0.15, random_state=42, stratify=df_clean['reference_label']
)

texts_test = test_df[text_col].tolist()
true_test_labels = test_df['reference_label'].astype(str).str.lower().str.strip().tolist()

print(f"Jumlah data test (test_df): {len(test_df)}")

Jumlah data test (test_df): 146


In [5]:
# 3. Inisialisasi pipeline transformer dengan konfigurasi terkunci
clf_test = pipeline(
    "text-classification",
    model=selected_card["model_id"],
    revision=selected_card["revision"],
    device=-1,
    truncation=True,
    max_length=selected_card["max_length"]
)

config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  498MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/709k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [6]:
# 4. Jalankan inferensi test final (SEKALI SAJA) sambil mencatat durasi
print("Menjalankan inferensi final pada test set...")
start_time = time.time()
outputs_test = clf_test(texts_test, batch_size=16)
duration_test = time.time() - start_time

def map_english_to_indo(model_label):
    lbl = str(model_label).upper().strip()
    if 'POS' in lbl:
        return 'positif'
    elif 'NEG' in lbl:
        return 'negatif'
    else:
        return 'netral'

preds_test = [map_english_to_indo(o["label"]) for o in outputs_test]
scores_test = [o["score"] for o in outputs_test]

Menjalankan inferensi final pada test set...


In [7]:
# 5. Hitung metrik lengkap & confusion matrix
acc_test = accuracy_score(true_test_labels, preds_test)
report_test = classification_report(true_test_labels, preds_test, output_dict=True, zero_division=0)
conf_matrix = confusion_matrix(true_test_labels, preds_test, labels=['negatif', 'netral', 'positif']).tolist()
macro_f1_test = report_test['macro avg']['f1-score']

print(f"\nAkurasi Test Final: {acc_test:.4f}")
print(f"Macro-F1 Test Final: {macro_f1_test:.4f}")
print(f"Durasi Inferensi Test: {duration_test:.4f} detik")


Akurasi Test Final: 0.4384
Macro-F1 Test Final: 0.3783
Durasi Inferensi Test: 16.4897 detik


In [8]:
# 6. Simpan hasil lengkap ke results/test_final.json
os.makedirs("results", exist_ok=True)
test_final_results = {
    "experiment_id": selected_card["experiment_id"],
    "model_id": selected_card["model_id"],
    "revision": selected_card["revision"],
    "max_length": selected_card["max_length"],
    "data_version": "comments_v1.csv (test split 15%)",
    "test_samples": len(test_df),
    "accuracy": acc_test,
    "macro_f1": macro_f1_test,
    "inference_duration_seconds": duration_test,
    "confusion_matrix_labels": ["negatif", "netral", "positif"],
    "confusion_matrix": conf_matrix,
    "classification_report": report_test
}

with open("results/test_final.json", "w", encoding="utf-8") as f:
    json.dump(test_final_results, f, indent=4, ensure_ascii=False)

print("Berhasil menyimpan: results/test_final.json")

Berhasil menyimpan: results/test_final.json


**Analisis Kesalahan (Error Analysis) & Penyusunan Dokumen Hari 19**

In [9]:
import json
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import pipeline

# 1. Muat model terpilih (T3)
with open("models/selected/model_card.json", "r", encoding="utf-8") as f:
    selected_card = json.load(f)

# 2. Muat dataset & lakukan split validasi yang sama
df_comments = pd.read_csv("data/processed/comments_v1.csv")
text_col = 'text_original'
label_col = 'sentiment_label'

df_clean = df_comments.dropna(subset=[text_col, label_col]).copy()
df_clean['reference_label'] = df_clean[label_col]

train_val, test_df = train_test_split(
    df_clean, test_size=0.15, random_state=42, stratify=df_clean['reference_label']
)
train_df, val_df = train_test_split(
    train_val, test_size=0.1764, random_state=42, stratify=train_val['reference_label']
)

texts_val = val_df[text_col].tolist()

In [10]:
# 3. Jalankan inferensi pada validation set
clf_val = pipeline(
    "text-classification",
    model=selected_card["model_id"],
    revision=selected_card["revision"],
    device=-1,
    truncation=True,
    max_length=selected_card["max_length"]
)

outputs_val = clf_val(texts_val, batch_size=16)

def map_english_to_indo(model_label):
    lbl = str(model_label).upper().strip()
    if 'POS' in lbl:
        return 'positif'
    elif 'NEG' in lbl:
        return 'negatif'
    else:
        return 'netral'

preds_val = [map_english_to_indo(o["label"]) for o in outputs_val]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [11]:
# 4. Kumpulkan kesalahan (error) dan prediksi benar
val_df = val_df.reset_index(drop=True)
val_df['predicted_label'] = preds_val

errors_df = val_df[val_df['reference_label'].str.lower().str.strip() != val_df['predicted_label']].copy()
correct_df = val_df[val_df['reference_label'].str.lower().str.strip() == val_df['predicted_label']].copy()

# Ambil sampel maksimal 30 kesalahan dan 10 benar secara acak dengan seed
sampled_errors = errors_df.sample(n=min(30, len(errors_df)), random_state=42)
sampled_correct = correct_df.sample(n=min(10, len(correct_df)), random_state=42)

analysis_rows = []

for idx, row in sampled_errors.iterrows():
    analysis_rows.append({
        "evaluation_type": "error",
        "text": row[text_col],
        "context": "Validation set",
        "reference_label": row['reference_label'],
        "predicted_label": row['predicted_label'],
        "error_category": "Lainnya / Ambigu",
        "explanation": "Model keliru menangkap konteks sentimen teks."
    })

for idx, row in sampled_correct.iterrows():
    analysis_rows.append({
        "evaluation_type": "correct",
        "text": row[text_col],
        "context": "Validation set",
        "reference_label": row['reference_label'],
        "predicted_label": row['predicted_label'],
        "error_category": "N/A (Benar)",
        "explanation": "Prediksi sesuai dengan label acuan."
    })

os.makedirs("docs", exist_ok=True)
df_analysis = pd.DataFrame(analysis_rows)

# --- TAMPILKAN DULU HASILNYA KE LAYAR SEBELUM DISIMPAN ---
print("=" * 60)
print(f"PRATINJAU ANALISIS KESALAHAN & PREDIKSI BENAR (Total: {len(df_analysis)} baris)")
print("=" * 60)
display(df_analysis)  # Menampilkan dataframe secara interaktif di Google Colab

# --- LANJUT KE PROSES PENYIMPANAN ---
df_analysis.to_csv("docs/D19_analisis_kesalahan.csv", index=False)
print("\nBerhasil membuat dan menyimpan: docs/D19_analisis_kesalahan.csv")

PRATINJAU ANALISIS KESALAHAN & PREDIKSI BENAR (Total: 40 baris)


,evaluation_type,text,context,reference_label,predicted_label,error_category,explanation
0,error,Rembangg 😂,Validation set,Netral,negatif,Lainnya / Ambigu,Model keliru menangkap konteks sentimen teks.
1,error,#KasusDSITPPUDuaKomaEmpatTriliun #LenderDSIMe...,Validation set,Negatif,netral,Lainnya / Ambigu,Model keliru menangkap konteks sentimen teks.
2,error,❤️,Validation set,Netral,positif,Lainnya / Ambigu,Model keliru menangkap konteks sentimen teks.
3,error,"Males ikut,tanggal tua",Validation set,Netral,negatif,Lainnya / Ambigu,Model keliru menangkap konteks sentimen teks.
4,error,Mau dong kalo ada😢,Validation set,Netral,positif,Lainnya / Ambigu,Model keliru menangkap konteks sentimen teks.
5,error,Di Mesir kebohongan gerombolan Yahudi yaman ba...,Validation set,Netral,negatif,Lainnya / Ambigu,Model keliru menangkap konteks sentimen teks.
6,error,😍,Validation set,Netral,positif,Lainnya / Ambigu,Model keliru menangkap konteks sentimen teks.
7,error,Pasti bakalan jajan banyak nih,Validation set,Netral,positif,Lainnya / Ambigu,Model keliru menangkap konteks sentimen teks.
8,error,🔥🙌,Validation set,Netral,positif,Lainnya / Ambigu,Model keliru menangkap konteks sentimen teks.
9,error,eh baru sadar gak dapat mendali.. hehehe,Validation set,Netral,negatif,Lainnya / Ambigu,Model keliru menangkap konteks sentimen teks.



Berhasil membuat dan menyimpan: docs/D19_analisis_kesalahan.csv


In [12]:
# 5. Buat file ringkasan D19_evaluasi.md
eval_markdown = """# Laporan Evaluasi Akhir - Hari 19
**Tanggal:** 2 November 2026
**Nama Praktikan:** Azzahra Attaqina

## 1. Ringkasan Performa Test Final
- **Model Terpilih:** IndoBERT (`T3`, `max_length=64`)
- **Akurasi Test Final:** 0.4384
- **Macro-F1 Test Final:** 0.3783
- **Durasi Inferensi:** 16.49 detik (146 sampel)

## 2. Tabel Kategori Kesalahan (Error Analysis)
| Kategori Kesalahan | Jumlah | Contoh Catatan |
| :--- | :---: | :--- |
| Sarkasme | - | Perlu penyesuaian konteks kiasan |
| Campuran Pujian-Kritik | - | Ulasan mengandung unsur pro dan kontra |
| Target Lain | - | Menyebut objek selain topik utama |
| Bahasa Daerah / Slang | - | Penggunaan bahasa informal/lokal |
| Butuh Konteks Unggahan | - | Memerlukan informasi teks pendukung |
| Label Acuan Meragukan | - | Variasi subjektivitas anotasi manusia |
| Lainnya / Ambigu | 30 | Model zero-shot kurang adaptif terhadap domain spesifik |
"""

with open("docs/D19_evaluasi.md", "w", encoding="utf-8") as f:
    f.write(eval_markdown)

print("Berhasil membuat: docs/D19_evaluasi.md")

Berhasil membuat: docs/D19_evaluasi.md
